# Mold Data — Quality Issue Summary by Family

Processes `data/mold_data.json` (schemaVersion 5.0) and exports an Excel report.

**Data quality issues:** I1 · I5 · I7 · I8 · I9 · I11 · I12 · I13 · I14
**v3 governance checks** (per `docs/reference/` standard): I15 · I16 · I17 · I18 · I19

**Output:** `data/dq_issues_by_family.xlsx`
- Sheet 1 `Flag Matrix` — instance counts per family × issue, sorted by most issues first
- One sheet per issue — affected families with per-component detail

## 1. Imports

In [1]:
import json
import re

import pandas as pd

from mold_v3_mapping import MOLD_ID_RE, parse_mold_id, WIDTH_VARIANT_RE

## 2. Load Data

In [2]:
with open('../data/mold_data.json', encoding='utf-8') as f:
    data = json.load(f)

assert data.get('schemaVersion') == '5.0', (
    f"Expected schemaVersion 5.0, got {data.get('schemaVersion')!r} — "
    "re-run data_processing.ipynb first."
)

families   = data['families']
vendor_ref = data['reference']['vendors']

print(f'Loaded {len(families)} mold families, {len(vendor_ref)} vendors')

Loaded 242 mold families, 74 vendors


## 3. Issue Detection Functions

In [3]:
def iter_assets(family):
    """Yield (mold_id, component, asset_index, asset) over the flat v5 structure."""
    for mold_id, comp in family.get('components', {}).items():
        for i, asset in enumerate(comp.get('assets', [])):
            yield mold_id, comp, i, asset


# ── Data quality checks (I1–I14) ──────────────────────────────────────────────

def check_null_vendor_id(family):
    """I1 — asset has null vendorId"""
    return [
        f'{mold_id} > asset[{i}]'
        for mold_id, comp, i, asset in iter_assets(family)
        if asset.get('vendorId') is None
    ]


def check_null_daily_output(family):
    """I5 — asset has null dailyOutput"""
    return [
        f'{mold_id} > asset[{i}]'
        for mold_id, comp, i, asset in iter_assets(family)
        if asset.get('capacity', {}).get('dailyOutput') is None
    ]


def check_empty_coverage(family):
    """I7 — asset has an empty sizeCoverage array (no base sizes recorded)"""
    return [
        f'{mold_id} > asset[{i}]'
        for mold_id, comp, i, asset in iter_assets(family)
        if not asset.get('sizeCoverage')
    ]


def check_null_coverage_qty(family):
    """I8 — sizeCoverage entry has null qty"""
    hits = []
    for mold_id, comp, i, asset in iter_assets(family):
        for j, entry in enumerate(asset.get('sizeCoverage', [])):
            if entry.get('qty') is None:
                hits.append(f'{mold_id} > asset[{i}] > sizeCoverage[{j}]: qty=null')
    return hits


def check_total_qty_mismatch(family):
    """I9 — totalMoldQty != sum(sizeCoverage qty values)"""
    hits = []
    for mold_id, comp, i, asset in iter_assets(family):
        total    = asset.get('totalMoldQty')
        coverage = asset.get('sizeCoverage', [])
        if total is not None and coverage:
            computed = sum(e.get('qty') or 0 for e in coverage)
            if computed != total:
                hits.append(
                    f'{mold_id} > asset[{i}]: declared={total}, sum_coverage_qty={computed}'
                )
    return hits


def check_style_soletype_missing(family):
    """I11 — style lists a soleType that has no component in this family"""
    hits = []
    comp_sole_types = {
        comp.get('soleType') for comp in family.get('components', {}).values()
    }
    for brand, styles in family.get('stylesUsingThisFamily', {}).items():
        for style in styles:
            for st in style.get('soleTypes', []):
                if st not in comp_sole_types:
                    hits.append(
                        f"{brand} > {style.get('styleName')}: '{st}' missing from components"
                    )
    return hits


def check_vendor_location_mismatch(family, vendor_ref):
    """I12 — vendor shortName country suffix contradicts location.code"""
    COUNTRY_CODES = {'BD', 'CN', 'VN', 'ID', 'KH', 'TH', 'MM', 'IN', 'PK', 'PH', 'TW', 'KR'}
    hits = []
    seen_vendors = set()
    for mold_id, comp, i, asset in iter_assets(family):
        vid = asset.get('vendorId')
        if not vid or vid not in vendor_ref or vid in seen_vendors:
            continue
        seen_vendors.add(vid)
        vdata = vendor_ref[vid]
        loc_code = (vdata.get('location') or {}).get('code', '')
        short_name = vdata.get('shortName', '')
        suffix = short_name.split('-')[-1][:2].upper() if '-' in short_name else ''
        if suffix in COUNTRY_CODES and suffix != loc_code:
            hits.append(
                f"{mold_id} > asset[{i}]: vendor {vid} "
                f"shortName='{short_name}' but location='{loc_code}'"
            )
    return hits


def check_duplicate_assets(family):
    """I13 — duplicate (vendorId, moldShopCode, initSeason) within the same component"""
    hits = []
    for mold_id, comp in family.get('components', {}).items():
        seen = {}
        for i, asset in enumerate(comp.get('assets', [])):
            key = (asset.get('vendorId'), asset.get('moldShopCode'), asset.get('initSeason'))
            if key in seen:
                hits.append(
                    f'{mold_id}: asset[{seen[key]}] duplicates asset[{i}] — {key}'
                )
            else:
                seen[key] = i
    return hits


def check_orphan_family(family):
    """I14 — no style references this mold family"""
    if not family.get('stylesUsingThisFamily'):
        return ['No styles reference this family']
    return []


# ── v3 governance checks (I15–I19, docs/reference/ §9) ───────────────────────

def check_invalid_mold_id(family):
    """I15 — component key is not a governed Mold ID, or disagrees with stored fields"""
    hits = []
    family_code = family.get('moldFamily')
    for mold_id, comp in family.get('components', {}).items():
        parsed = parse_mold_id(mold_id)
        if parsed is None:
            hits.append(f"{mold_id}: does not match {{Family}}-{{Type}}-{{Position}}(-B)")
            continue
        if parsed['family'] != family_code:
            hits.append(f"{mold_id}: family segment '{parsed['family']}' != '{family_code}'")
        if comp.get('moldId') != mold_id:
            hits.append(f"{mold_id}: stored moldId '{comp.get('moldId')}' != key")
        for field in ('type', 'position', 'stage'):
            if comp.get(field) != parsed[field]:
                hits.append(
                    f"{mold_id}: stored {field}='{comp.get(field)}' != '{parsed[field]}' from key"
                )
    return hits


def check_missing_pri(family):
    """I16 — a (family, type) has components but no PRI position"""
    hits = []
    positions_by_type = {}
    for comp in family.get('components', {}).values():
        positions_by_type.setdefault(comp.get('type'), set()).add(comp.get('position'))
    for type_, positions in sorted(positions_by_type.items(), key=str):
        if 'PRI' not in positions:
            hits.append(f"{type_}: positions {sorted(positions)} exist but PRI is missing")
    return hits


def check_blocker_without_default(family):
    """I17 — a -B blocker Mold ID has no paired non-B Mold ID"""
    hits = []
    components = family.get('components', {})
    for mold_id, comp in components.items():
        if comp.get('stage') == 'Blocker':
            paired = mold_id[:-2] if mold_id.endswith('-B') else mold_id
            if paired not in components:
                hits.append(f"{mold_id}: paired default '{paired}' is missing")
    return hits


def check_ot_not_pri(family):
    """I18 — an OT component uses a position other than PRI"""
    return [
        f"{mold_id}: OT must always be PRI, found {comp.get('position')}"
        for mold_id, comp in family.get('components', {}).items()
        if comp.get('type') == 'OT' and comp.get('position') != 'PRI'
    ]


def check_nonstandard_family_code(family):
    """I19 — family code is frozen-legacy or carries suspicious suffixes

    Width/gender variants (-E/-4E/-M/-W) are legitimate separate families
    and are allow-listed. Trailing -1/-2/-3 or -A/-B/-C suggest a legacy
    Revision / Mold Set suffix that belongs on the asset record instead.
    """
    code = family.get('moldFamily', '')
    hits = []
    if code != code.strip():
        hits.append(f"'{code}': contains leading/trailing whitespace")
    if family.get('isLegacyCode'):
        hits.append(f"'{code}': non-standard code (frozen legacy — retain, do not replicate)")
    elif not WIDTH_VARIANT_RE.match(code) and re.search(r'-(\d|[A-D])$', code):
        hits.append(
            f"'{code}': trailing suffix looks like a Revision/Mold Set marker — "
            "should be an asset attribute, not part of the family code"
        )
    return hits

## 4. Build Per-Family Summary

In [4]:
CHECKS = [
    ('I1_null_vendorId',        lambda f: check_null_vendor_id(f)),
    ('I5_null_dailyOutput',     lambda f: check_null_daily_output(f)),
    ('I7_empty_coverage',       lambda f: check_empty_coverage(f)),
    ('I8_null_coverage_qty',    lambda f: check_null_coverage_qty(f)),
    ('I9_total_qty_mismatch',   lambda f: check_total_qty_mismatch(f)),
    ('I11_style_stype_missing', lambda f: check_style_soletype_missing(f)),
    ('I12_vendor_loc_mismatch', lambda f: check_vendor_location_mismatch(f, vendor_ref)),
    ('I13_duplicate_assets',    lambda f: check_duplicate_assets(f)),
    ('I14_orphan_family',       lambda f: check_orphan_family(f)),
    ('I15_invalid_mold_id',     lambda f: check_invalid_mold_id(f)),
    ('I16_missing_pri',         lambda f: check_missing_pri(f)),
    ('I17_blocker_unpaired',    lambda f: check_blocker_without_default(f)),
    ('I18_ot_not_pri',          lambda f: check_ot_not_pri(f)),
    ('I19_nonstd_family_code',  lambda f: check_nonstandard_family_code(f)),
]

ISSUE_LABELS = {
    'I1_null_vendorId':        'I1 · Null vendorId',
    'I5_null_dailyOutput':     'I5 · Null dailyOutput',
    'I7_empty_coverage':       'I7 · Empty sizeCoverage',
    'I8_null_coverage_qty':    'I8 · Null sizeCoverage qty',
    'I9_total_qty_mismatch':   'I9 · totalMoldQty mismatch',
    'I11_style_stype_missing': 'I11 · Style soleType missing',
    'I12_vendor_loc_mismatch': 'I12 · Vendor location mismatch',
    'I13_duplicate_assets':    'I13 · Duplicate assets',
    'I14_orphan_family':       'I14 · Orphan family',
    'I15_invalid_mold_id':     'I15 · Invalid Mold ID',
    'I16_missing_pri':         'I16 · Missing PRI',
    'I17_blocker_unpaired':    'I17 · Blocker without default',
    'I18_ot_not_pri':          'I18 · OT not PRI',
    'I19_nonstd_family_code':  'I19 · Non-standard family code',
}

SHEET_NAMES = {
    'I1_null_vendorId':        'I1 - Null vendorId',
    'I5_null_dailyOutput':     'I5 - Null dailyOutput',
    'I7_empty_coverage':       'I7 - Empty sizeCoverage',
    'I8_null_coverage_qty':    'I8 - Null sizeCoverage qty',
    'I9_total_qty_mismatch':   'I9 - TotalQty mismatch',
    'I11_style_stype_missing': 'I11 - Style soleType miss',
    'I12_vendor_loc_mismatch': 'I12 - Vendor loc mismatch',
    'I13_duplicate_assets':    'I13 - Duplicate assets',
    'I14_orphan_family':       'I14 - Orphan family',
    'I15_invalid_mold_id':     'I15 - Invalid Mold ID',
    'I16_missing_pri':         'I16 - Missing PRI',
    'I17_blocker_unpaired':    'I17 - Blocker unpaired',
    'I18_ot_not_pri':          'I18 - OT not PRI',
    'I19_nonstd_family_code':  'I19 - Nonstd family code',
}

rows = []
for family_code, family in families.items():
    row = {'family': family_code, 'total_issues': 0}
    for col, fn in CHECKS:
        hits = fn(family)
        row[col] = len(hits)
        row[col + '_detail'] = ' | '.join(hits) if hits else ''
        row['total_issues'] += (1 if hits else 0)
    rows.append(row)

df = pd.DataFrame(rows).set_index('family')
count_cols = [c for c, _ in CHECKS]

print(f'Families with at least one issue: {(df["total_issues"] > 0).sum()} / {len(df)}')

Families with at least one issue: 163 / 242


## 5. Export to Excel

In [5]:
output_path = '../data/dq_issues_by_family.xlsx'

# Flag matrix: families with issues, sorted by most issue types hit
_sorted_idx = (
    df[df['total_issues'] > 0]
    .sort_values('total_issues', ascending=False)
    .index
)
flag_df = (
    df.loc[_sorted_idx, ['total_issues'] + count_cols]
    .rename(columns={'total_issues': 'Total Issues', **ISSUE_LABELS})
)

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:

    # Sheet 1 — flag matrix
    flag_df.to_excel(writer, sheet_name='Flag Matrix')

    # One sheet per issue (skipped if no families are affected)
    for col, _ in CHECKS:
        affected = df[df[col] > 0][[col, col + '_detail']].copy()
        if affected.empty:
            continue
        affected = (
            affected
            .rename(columns={col: 'instance_count', col + '_detail': 'detail'})
            .sort_values('instance_count', ascending=False)
        )
        affected.index.name = 'family'
        affected.to_excel(writer, sheet_name=SHEET_NAMES[col])

print(f'Exported: {output_path}')
print(f'  Flag Matrix : {len(flag_df)} families')
for col, _ in CHECKS:
    n = (df[col] > 0).sum()
    if n:
        print(f'  {SHEET_NAMES[col]:<28}: {n} families')

Exported: ../data/dq_issues_by_family.xlsx
  Flag Matrix : 163 families
  I1 - Null vendorId          : 4 families
  I5 - Null dailyOutput       : 22 families
  I11 - Style soleType miss   : 8 families
  I13 - Duplicate assets      : 2 families
  I14 - Orphan family         : 150 families
  I16 - Missing PRI           : 3 families
  I19 - Nonstd family code    : 2 families
